# Normalization & adjusted EBITDA

Walk from **reported EBITDA** to **adjusted EBITDA** using `NormalizationConfig` with fixed add-backs, **percentage-of-revenue** fees, and **capped** synergies.

## Concept

`NormalizationConfig` is JSON-serializable: each adjustment has a `type` of `fixed` (per-period map), `percentage_of_node`, or `formula`. Caps reference a **base node** and a **percentage limit**. Python exposes `normalize(result, config)` which returns a JSON string; parse it with `json.loads` to get Python dict rows.

**Cap semantics:** When an adjustment has a `cap`, the limit is computed as `cap.value * running_adjusted_value` — meaning the cap is a percentage of the **cumulative adjusted total** (base + all prior adjustments), not the raw base value. This prevents runaway add-backs while staying transparent in the output.

In [1]:
import json

import sys
sys.path.insert(0, "../..")

from _shared import series
from finstack_quant.statements import (
    Evaluator,
    ModelBuilder,
    NormalizationConfig,
    normalize,
)

PERIODS = ["2025Q1", "2025Q2"]

b = ModelBuilder("norm-demo")
b.periods("2025Q1..Q2", None)
b.value("revenue", series(PERIODS, [1000.0, 1100.0]))
b.value("ebitda", series(PERIODS, [120.0, 135.0]))
spec = b.build()
res = Evaluator().evaluate(spec)

cfg = NormalizationConfig("ebitda")
print("Empty config:", cfg, "| adjustments:", cfg.adjustment_count)


Empty config: NormalizationConfig(target="ebitda", adjustments=0) | adjustments: 0


In [2]:
import json
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/normalization_and_adjustments.json").read_text())

print("Normalization JSON config + engine")
cfg_json = _NOTEBOOK_DATA['cfg_json']
full_cfg = NormalizationConfig.from_json(json.dumps(cfg_json))
print("Loaded adjustments:", full_cfg.adjustment_count)

rows = json.loads(normalize(res, full_cfg))
for row in rows:
    print(f"\n{row['period']}:  base={row['base_value']:.1f}  final={row['final_value']:.1f}")
    for adj in row["adjustments"]:
        capped = " (CAPPED)" if adj["is_capped"] else ""
        print(f"  + {adj['name']:45s}  raw={adj['raw_amount']:6.1f}  applied={adj['capped_amount']:6.1f}{capped}")


Normalization JSON config + engine
Loaded adjustments: 3

2025Q1:  base=120.0  final=166.0
  + Restructuring add-back                         raw=   8.0  applied=   8.0
  + Mgmt fee (% of revenue)                        raw=  20.0  applied=  20.0
  + Synergies (capped at 15% of running adjusted EBITDA)  raw=  30.0  applied=  18.0 (CAPPED)

2025Q2:  base=135.0  final=182.2
  + Restructuring add-back                         raw=   5.0  applied=   5.0
  + Mgmt fee (% of revenue)                        raw=  22.0  applied=  22.0
  + Synergies (capped at 15% of running adjusted EBITDA)  raw=  25.0  applied=  20.2 (CAPPED)


## Takeaways

- Keep a **machine-readable adjustment policy** (`from_json`) next to human memos for audit.
- **Percentage fees** and **caps** prevent runaway add-backs while staying transparent in the per-period breakdown returned by `normalize`.
- Merge normalized series back into models with downstream nodes if you need **forecast adjusted EBITDA**.